# Assignment 2B — RAG Pipeline (Manufacturing & Technical Standards)

**Course:** Large Language Models for Generative AI (S1-26_AIMLZG536)  **Group:** 78

Builds a small, complete Retrieval-Augmented Generation (RAG) system on the Assignment 1 domain corpus:

```
12 cleaned .txt docs ──► chunk ──► embed ──► FAISS (dense) ─┐
                              └──► BM25 (keyword) ──────────┼─► RRF hybrid ─► cross-encoder rerank ─► top-3 chunks
                                                                                                         │
                         Assignment 1 model (TinyLlama-1.1B + QLoRA Adapter B) ◄── grounded prompt ◄──────┘
                                         │
                                         └──► LLM judge (correct? supported?)
```

| Part | What we do | Marks |
|---|---|---|
| A | Dense search + choose a chunk size | 5 |
| B | Dense vs BM25 vs Hybrid (RRF) | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded generation + LLM judge | 7 |

**Reused from Assignment 1:** the 12 cleaned domain `.txt` files, the 10-pair evaluation set (`eval_dataset.jsonl`, seed-42 split), and the fine-tuned TinyLlama-1.1B + QLoRA Adapter B.

---
## Step 0 — Environment setup
Parts A–C run on CPU or GPU. Part D loads LLMs and needs a GPU (Prayogshala / Colab).

In [ ]:
%pip install -q sentence-transformers faiss-cpu rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.8 MB/s eta 0:00:00


In [ ]:
import os, re, json, time, random, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import torch

random.seed(42); np.random.seed(42); torch.manual_seed(42)
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only")

# Find the corpus folder whichever platform we are on
CANDIDATES = [Path("domain_corpus"), Path("/home/jovyan/domain_corpus"), Path("/content/domain_corpus")]
CORPUS_DIR = next((p for p in CANDIDATES if p.exists() and any(p.glob("*.txt"))), None)
assert CORPUS_DIR is not None, "Upload the 12 cleaned .txt files into a folder named domain_corpus/"
print("Corpus folder:", CORPUS_DIR.resolve())

Device: cuda | Tesla T4
Corpus folder: /content/domain_corpus


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

# Create the domain_corpus directory if it doesn't exist
corpus_path = Path('domain_corpus')
corpus_path.mkdir(exist_ok=True)
print(f"Folder '{corpus_path.resolve()}' is ready. Please upload your 12 cleaned .txt files here.")

Folder '/content/domain_corpus' is ready. Please upload your 12 cleaned .txt files here.


### 0.1 Load the document collection
We reuse the cleaned Assignment 1 files as-is. Only whitespace is collapsed — we deliberately do **not** fix the OCR noise
(e.g. `management-ied`, `weii`) because real RAG systems must cope with imperfect text, and the answer spans below are
copied verbatim from this text.

In [ ]:
def normalize_ws(text: str) -> str:
    # Collapse all runs of whitespace (newlines, tabs, double spaces) to a single space.
    return re.sub(r"\s+", " ", text).strip()

docs = {p.name: normalize_ws(p.read_text(encoding="utf-8", errors="ignore"))
        for p in sorted(CORPUS_DIR.glob("*.txt"))}

corpus_stats = pd.DataFrame([{"document": n, "characters": len(t), "words": len(t.split())} for n, t in docs.items()])
print(f"{len(docs)} documents | {corpus_stats.characters.sum():,} characters | {corpus_stats.words.sum():,} words")
corpus_stats

12 documents | 2,448,363 characters | 426,339 words


,document,characters,words
0,NIST_ISO9000_Quality_Systems.txt,48600,7533
1,NIST_Malcolm_Baldrige_Quality.txt,39432,6001
2,NIST_Manufacturing_Handbook.txt,1092303,210709
3,NIST_Manufacturing_Machinery_Maintenance.txt,113122,17206
4,NIST_Measurement_Uncertainty.txt,87256,14294
5,NIST_QIF_Quality_Information_Framework.txt,74356,10408
6,NIST_Quality_Assurance_Manufacturing.txt,38236,5552
7,NIST_Quality_Systems_Handbook.txt,124549,20636
8,OSHA_Ergonomics_Workplace.txt,31772,5484
9,OSHA_General_Industry_Standards.txt,539628,88902


### 0.2 The 10 evaluation queries and their answer spans
The queries and expected answers are the 10-pair evaluation split from Assignment 1 (48 curated pairs, 80/20 split, `random.seed(42)`).

**Answer span** = the sentence or short passage in the corpus that answers the query. A retrieved chunk is a **hit** if it contains the span.
Each span was found by searching the query's source document and copied verbatim (OCR errors included). Spans are kept short
(13–27 tokens) so that each one can fit inside even a 100-token chunk.

In [ ]:
EVAL = [
 {"qid": 1, "source": "NIST_Malcolm_Baldrige_Quality.txt",
  "query": "How did Malcolm Baldrige quality concepts influence the STEP model qualification approach?",
  "expected": "The STEP qualification effort used several ideas drawn from the Malcolm Baldrige National Quality Award criteria. These ideas encouraged measurable quality thresholds, links between processes and results, continuous improvement, comparison with established performance levels, and management involvement in producing high-quality information models.",
  "span": "Many basic concepts of the Malcolm Baldrige National Quality Award were used in building the foundation for this new approach to standards development"},
 {"qid": 2, "source": "OSHA_Ergonomics_Workplace.txt",
  "query": "How can workstation design reduce excessive reaching and awkward upper-body posture?",
  "expected": "Workstations can be arranged so materials and products are positioned closer to workers. Examples include conveyor diverters, appropriately located chutes or cut-outs, and correctly adjusted work platforms. These changes can reduce long reaches, elevated arms, wrist deviation, forward bending, and other stressful postures.",
  "span": "This conveyor has diverters in place that push the products closer to the workers. This reduces reaches for the employees."},
 {"qid": 3, "source": "NIST_Malcolm_Baldrige_Quality.txt",
  "query": "What three kinds of evidence can be used to assess results under the Baldrige-inspired approach described in the report?",
  "expected": "The report describes results as being assessed through improvement trends, current performance levels, and benchmarks or other comparisons with external performance. Together these forms of evidence show whether quality is improving, what level has been reached, and how performance compares with appropriate leaders.",
  "span": "based upon one or more of three factors: (1) improvement trends; (2) current levels; and (3) benchmarks"},
 {"qid": 4, "source": "NIST_Manufacturing_Handbook.txt",
  "query": "Why are statistical methods useful when analysing manufacturing experiments?",
  "expected": "Statistical methods provide a systematic way to summarize experimental observations, describe the amount of variation in measured results, and judge whether apparent differences are meaningful. This helps manufacturing engineers draw conclusions from data rather than relying only on individual observations or subjective impressions.",
  "span": "One of the most frequent uses of statistics is in testing for differences."},
 {"qid": 5, "source": "NIST_Manufacturing_Machinery_Maintenance.txt",
  "query": "What is the difference between reactive maintenance and more advanced maintenance strategies?",
  "expected": "Reactive maintenance responds after machinery has already failed or developed a problem. Preventive and predictive approaches attempt to reduce failures by maintaining equipment before breakdowns occur or by using information about equipment condition to guide maintenance decisions, thereby reducing avoidable downtime and production losses.",
  "span": "is maintenance done, typically, after equipment has failed to produce a product within desired quality or production targets"},
 {"qid": 6, "source": "NIST_Manufacturing_Machinery_Maintenance.txt",
  "query": "What costs are included when analysing direct and indirect machinery-maintenance impacts?",
  "expected": "Direct maintenance costs include labour and materials used for maintenance and repair. The broader economic analysis also considers cascading damage following breakdowns, inventory increases caused by uncertainty or disruption, unplanned downtime, defects, lost sales, and other losses associated with faults and failures.",
  "span": "Direct maintenance and repair costs include the cost of labor and materials along with cascading effects"},
 {"qid": 7, "source": "NIST_Measurement_Uncertainty.txt",
  "query": "What is the difference between Type A and Type B evaluations of uncertainty?",
  "expected": "A Type A evaluation determines an uncertainty component through statistical analysis of repeated observations. A Type B evaluation uses other available information, such as calibration data, previous measurements, instrument specifications, reference data, or scientific judgement, rather than relying on repeated statistical observations alone.",
  "span": "The evaluation of uncertainty by the statistical analysis of series of observations is termed a Type A evaluation"},
 {"qid": 8, "source": "NIST_ISO9000_Quality_Systems.txt",
  "query": "What common elements are included in the ISO 9001, ISO 9002, and ISO 9003 quality-system models?",
  "expected": "The ISO 9000 quality-system models share requirements for an effective quality system, valid measurements, regularly calibrated measuring and testing equipment, suitable statistical techniques, product identification and traceability, adequate records, inspection and testing, proper handling and storage, and control of nonconforming items.",
  "span": "Common elements in ISO 9001, 9002, and 9003 include the need for; an effective quality system"},
 {"qid": 9, "source": "NIST_Malcolm_Baldrige_Quality.txt",
  "query": "Why is management involvement important in a quality-improvement program?",
  "expected": "The report emphasizes that successful quality improvement should be management-led and customer-oriented. Management establishes expectations, provides direction and resources, supports the use of quality metrics and thresholds, and creates the organizational conditions needed for persistent, systematic improvement rather than isolated corrective actions.",
  "span": "quality improvement programs must be management-ied and customer-oriented"},
 {"qid": 10, "source": "OSHA_Machine_Guarding_Standards.txt",
  "query": "What is the main purpose of machine guarding?",
  "expected": "Machine guarding protects operators and other employees from hazards created by moving machine parts and operations. Safeguards are intended to prevent workers from reaching dangerous areas such as points of operation, rotating components, nip points, and other locations where cutting, crushing, or entanglement injuries could occur.",
  "span": "Safeguarding is essential for protecting employees from needless and preventable injury."},
]
QUERIES = [e["query"] for e in EVAL]

# Hit test used in Parts A, B and C: case- and whitespace-insensitive substring match
def norm(s: str) -> str:
    return normalize_ws(s).lower()

def is_hit(chunk_text: str, span: str) -> bool:
    return norm(span) in norm(chunk_text)

# Sanity check: every span must exist (exactly once) in its source document
for e in EVAL:
    n = norm(docs[e["source"]]).count(norm(e["span"]))
    assert n >= 1, f"Q{e['qid']}: span not found in {e['source']}"
    e["span_occurrences"] = n

pd.DataFrame(EVAL)[["qid", "query", "source", "span", "span_occurrences"]]

,qid,query,source,span,span_occurrences
0,1,How did Malcolm Baldrige quality concepts influence the STEP model qualification approach?,NIST_Malcolm_Baldrige_Quality.txt,Many basic concepts of the Malcolm Baldrige National Quality Award were used in building the foundation for this new...,1
1,2,How can workstation design reduce excessive reaching and awkward upper-body posture?,OSHA_Ergonomics_Workplace.txt,This conveyor has diverters in place that push the products closer to the workers. This reduces reaches for the empl...,1
2,3,What three kinds of evidence can be used to assess results under the Baldrige-inspired approach described in the rep...,NIST_Malcolm_Baldrige_Quality.txt,based upon one or more of three factors: (1) improvement trends; (2) current levels; and (3) benchmarks,1
3,4,Why are statistical methods useful when analysing manufacturing experiments?,NIST_Manufacturing_Handbook.txt,One of the most frequent uses of statistics is in testing for differences.,1
4,5,What is the difference between reactive maintenance and more advanced maintenance strategies?,NIST_Manufacturing_Machinery_Maintenance.txt,"is maintenance done, typically, after equipment has failed to produce a product within desired quality or production...",1
5,6,What costs are included when analysing direct and indirect machinery-maintenance impacts?,NIST_Manufacturing_Machinery_Maintenance.txt,Direct maintenance and repair costs include the cost of labor and materials along with cascading effects,1
6,7,What is the difference between Type A and Type B evaluations of uncertainty?,NIST_Measurement_Uncertainty.txt,The evaluation of uncertainty by the statistical analysis of series of observations is termed a Type A evaluation,1
7,8,"What common elements are included in the ISO 9001, ISO 9002, and ISO 9003 quality-system models?",NIST_ISO9000_Quality_Systems.txt,"Common elements in ISO 9001, 9002, and 9003 include the need for; an effective quality system",1
8,9,Why is management involvement important in a quality-improvement program?,NIST_Malcolm_Baldrige_Quality.txt,quality improvement programs must be management-ied and customer-oriented,1
9,10,What is the main purpose of machine guarding?,OSHA_Machine_Guarding_Standards.txt,Safeguarding is essential for protecting employees from needless and preventable injury.,1


---
# Part A — Dense search and chunk size (5 marks)

**Dense retrieval in one paragraph.** An embedding model maps any text to a fixed-length vector so that texts with similar
*meaning* land close together. We embed every chunk once (offline), store the vectors in an index, and at query time embed the
query and return the chunks whose vectors are most similar. If all vectors are L2-normalised, the inner product equals cosine
similarity, so a FAISS `IndexFlatIP` (exact, brute-force inner-product search) gives exact cosine-similarity search.

### Embedding model choice and its input limit
The assignment asks for chunk sizes of 100 and 400 tokens. `all-MiniLM-L6-v2` truncates input at **256** tokens, so a
400-token chunk would lose its last ~144 tokens when embedded (they would still be sent to the LLM, but would be invisible to search).
**We therefore switched to a model with a longer limit: `BAAI/bge-small-en-v1.5`** — 512-token limit, 384-dim vectors,
33M parameters, so it is just as cheap to run. Tokens are counted with this model's own tokenizer.

BGE models are trained with an instruction prefix for short queries
(`"Represent this sentence for searching relevant passages: "`); passages are embedded without it.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX   = "Represent this sentence for searching relevant passages: "
EMBED_BATCH    = 64

embedder  = SentenceTransformer(EMBED_MODEL_ID, device=DEVICE)
tokenizer = embedder.tokenizer
MAX_TOKENS = embedder.max_seq_length            # limit applied when embedding (incl. [CLS]/[SEP])
EMBED_DIM  = (embedder.get_embedding_dimension() if hasattr(embedder, "get_embedding_dimension")
              else embedder.get_sentence_embedding_dimension())

print("Embedding model :", EMBED_MODEL_ID)
print("Max input length:", MAX_TOKENS, "tokens")
print("Vector size     :", EMBED_DIM)

def count_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

## A1. Build dense search (2 marks)

### A1.1 — One reusable dense-search component
`DenseIndex` embeds a list of chunks, builds a FAISS `IndexFlatIP`, and `search(query, k)` returns the top-k chunks with scores.
The same class is reused for every chunk size in A2 and again in Parts B and C.

In [ ]:
class DenseIndex:
    # Embed chunks -> FAISS IndexFlatIP (cosine via normalised vectors) -> top-k search.

    def __init__(self, chunks, model=embedder, batch_size=EMBED_BATCH, query_prefix=QUERY_PREFIX):
        self.chunks, self.model, self.batch_size, self.query_prefix = chunks, model, batch_size, query_prefix
        texts = [c["text"] for c in chunks]

        t0 = time.perf_counter()
        vecs = model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=False).astype("float32")
        t1 = time.perf_counter()
        self.index = faiss.IndexFlatIP(vecs.shape[1])
        self.index.add(vecs)
        t2 = time.perf_counter()

        self.embed_seconds, self.add_seconds = t1 - t0, t2 - t1
        self.build_seconds = t2 - t0

    def search(self, query: str, k: int = 5):
        q = self.model.encode([self.query_prefix + query], normalize_embeddings=True,
                              convert_to_numpy=True).astype("float32")
        scores, ids = self.index.search(q, k)
        return [{**self.chunks[i], "score": float(s), "rank": r}
                for r, (i, s) in enumerate(zip(ids[0], scores[0]), start=1)]

## A2. Choose a chunk size (3 marks)

### Fixed-size chunking with the embedding model's tokenizer
Each document is tokenized once with `return_offsets_mapping=True`. That gives, for every token, its character position in the
original text, so a window of N tokens can be cut directly from the **original** text (no decode step, so casing and
punctuation are preserved). Chunks never cross document boundaries, and there is no overlap, as the assignment specifies.

Both sizes fit the model: 100 + 2 and 400 + 2 special tokens are ≤ 512, so nothing is truncated.

In [ ]:
def chunk_fixed(docs: dict, size: int):
    # Split every document into consecutive, non-overlapping windows of `size` tokens.
    chunks = []
    for name, text in docs.items():
        enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True, verbose=False)
        offsets = enc["offset_mapping"]
        for start in range(0, len(offsets), size):
            window = offsets[start:start + size]
            chunks.append({
                "chunk_id": len(chunks),
                "doc": name,
                "text": text[window[0][0]:window[-1][1]],
                "n_tokens": len(window),
            })
    return chunks

CHUNK_SIZES = [100, 400]
chunk_sets = {size: chunk_fixed(docs, size) for size in CHUNK_SIZES}

for size, chunks in chunk_sets.items():
    assert max(c["n_tokens"] for c in chunks) + 2 <= MAX_TOKENS
    print(f"{size:>3}-token chunks: {len(chunks):>5} chunks | example from {chunks[1]['doc']}:")
    print(textwrap.indent(textwrap.fill(chunks[1]["text"][:300] + " ...", 110), "    "))

100-token chunks:  6386 chunks | example from NIST_ISO9000_Quality_Systems.txt:
    COMMERCE Robert A. Mosbacher, Secretary NATIONAL INSTITUTE OF STANDARDS AND TECHNOLOGY John W. Lyons, Director
    NIST NISTIR 4721 Questions and Answers on Quality, the ISO 9000 Standard Series, Quality System Registration,
    and Related Issues Maureen Breitenberg U.S. DEPARTMENT OF COMMERCE National Ins ...
400-token chunks:  1601 chunks | example from NIST_ISO9000_Quality_Systems.txt:
    s Office of European Community Affairs; Mr. Charles Hyer, the Marley Organization; Mr. Donald Mackay, Air-
    Conditioning and Refrigeration Institute; Dr. Curt Reimann and Dr. Robert Chapman, NIST's Office of Quality
    Programs; Patricia Kopp, American Society for Quality Control; and NIST's Standards C ...


**Can every span be found at each size?** With no overlap, a span that happens to straddle a chunk boundary is split in two,
and *no* chunk can ever contain it. Checking this up front tells us the best hit rate each chunk size could possibly reach.

In [ ]:
reachable = pd.DataFrame({
    f"{size} tokens": ["yes" if any(is_hit(c["text"], e["span"]) for c in chunks) else "SPLIT"
                       for e in EVAL]
    for size, chunks in chunk_sets.items()
}, index=[f"Q{e['qid']}" for e in EVAL])
print("Span contained intact in at least one chunk:")
reachable.T

Span contained intact in at least one chunk:


,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10
100 tokens,yes,SPLIT,yes,yes,SPLIT,yes,yes,yes,yes,yes
400 tokens,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes


### A2.1 — Dense search at 100 vs 400 tokens (2 marks)
For each size: build the dense index (A1), retrieve the top 3 for each of the 10 queries, and compute
* **Hit rate@3** — share of queries where one of the top-3 chunks contains the answer span
* **Tokens sent per query** — combined token count of the top-3 chunks, averaged over the 10 queries

In [ ]:
from transformers import AutoTokenizer

# The generator (Assignment 1 TinyLlama) uses a different tokenizer from the embedder, so count its tokens too
GEN_MODEL_ID  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)

def count_gen_tokens(text: str) -> int:
    return len(gen_tokenizer(text, add_special_tokens=False)["input_ids"])

def hit_at_k(results, span, k):
    return any(is_hit(r["text"], span) for r in results[:k])

dense_indexes, a2_rows, a2_detail = {}, [], []
for size, chunks in chunk_sets.items():
    idx = DenseIndex(chunks)
    dense_indexes[size] = idx
    hits, tokens_sent, gen_tokens_sent = [], [], []
    for e in EVAL:
        top3 = idx.search(e["query"], k=3)
        hit_rank = next((r["rank"] for r in top3 if is_hit(r["text"], e["span"])), None)
        hits.append(hit_rank is not None)
        tokens_sent.append(sum(r["n_tokens"] for r in top3))
        gen_tokens_sent.append(sum(count_gen_tokens(r["text"]) for r in top3))
        a2_detail.append({"chunk size": size, "qid": e["qid"], "hit rank (top 3)": hit_rank or "-",
                          "top-1 doc": top3[0]["doc"], "correct doc": e["source"]})
    a2_rows.append({"Chunk size (tokens)": size,
                    "Total chunks": len(chunks),
                    "Hit rate@3 (%)": 100 * np.mean(hits),
                    "Tokens sent per query (top 3)": round(np.mean(tokens_sent), 1),
                    "TinyLlama tokens per query (avg)": round(np.mean(gen_tokens_sent), 1),
                    "TinyLlama tokens per query (max)": max(gen_tokens_sent),
                    "Index build time (s)": round(idx.build_seconds, 2)})

a2_table = pd.DataFrame(a2_rows)
a2_table

In [ ]:
# Per-query view: which queries hit, and at what rank? Did the top-1 chunk even come from the right document?
pd.DataFrame(a2_detail).pivot(index="qid", columns="chunk size", values=["hit rank (top 3)", "top-1 doc"])

hit rank (top 3)                                         top-1 doc  \
chunk size              100 400                                           100   
qid                                                                             
1                         2   -             NIST_Malcolm_Baldrige_Quality.txt   
2                         -   3                 OSHA_Ergonomics_Workplace.txt   
3                         2   2             NIST_Malcolm_Baldrige_Quality.txt   
4                         3   -               NIST_Manufacturing_Handbook.txt   
5                         -   -  NIST_Manufacturing_Machinery_Maintenance.txt   
6                         1   2  NIST_Manufacturing_Machinery_Maintenance.txt   
7                         -   3              NIST_Measurement_Uncertainty.txt   
8                         1   2              NIST_ISO9000_Quality_Systems.txt   
9                         1   3             NIST_Malcolm_Baldrige_Quality.txt   
10                        -   -           OSHA_Machine_Guarding_Standards.txt   

                                                          
chunk size                                           400  
qid                                                       
1                      NIST_Malcolm_Baldrige_Quality.txt  
2                          OSHA_Ergonomics_Workplace.txt  
3                      NIST_Malcolm_Baldrige_Quality.txt  
4                        NIST_Manufacturing_Handbook.txt  
5           NIST_Manufacturing_Machinery_Maintenance.txt  
6           NIST_Manufacturing_Machinery_Maintenance.txt  
7                       NIST_Measurement_Uncertainty.txt  
8                       NIST_ISO9000_Quality_Systems.txt  
9                    OSHA_General_Industry_Standards.txt  
10                   OSHA_Machine_Guarding_Standards.txt

**Observations from the A2 results**
* **Same hit rate@3 (60%), different queries.** 100-token chunks hit Q1, Q3, Q4, Q6, Q8, Q9; 400-token chunks hit Q2, Q3, Q6, Q7, Q8, Q9.
  Q2 is only reachable at 400 (its span is split at 100); Q7 needs the longer window.
* **Rank 1 is better with small chunks** — 3/10 at 100 tokens (Q6, Q8, Q9) vs 0/10 at 400. A small chunk's embedding is
  dominated by the answer sentence; a 400-token chunk mixes several topics, so its vector is "diluted" and lands at rank 2–3.
* **Right document, wrong passage.** Q5 and Q10 are missed at both sizes, yet the top-1 chunk comes from the correct document
  for 10/10 queries at 100 tokens and 9/10 at 400. Dense search finds the right *area*; the problem is ranking *within* it —
  exactly what hybrid search (Part B) and reranking (Part C) target.
* **Build time is about equal (~14 s)** even though 100-token chunks are 4× more numerous: the total number of tokens embedded
  is the same (~640k), only packed into more, shorter sequences.
* **Tokenizers differ.** One BGE token ≈ 1.25 TinyLlama tokens on this corpus (OCR noise and numbers split into many pieces),
  so "1,200 tokens" for the embedder is ~1,500 for the generator.

### A2.2 — Chosen chunk size (1 mark)
**We use 400-token chunks for Parts B–D.** Both sizes reach the same hit rate@3 (60%), but at 100 tokens the answer spans of
Q2 and Q5 are cut across chunk boundaries, capping that size at 80%, while every span stays intact at 400 tokens and each
chunk carries enough surrounding text for multi-part answers. The cost is 4× more context — about 1,200 embedding tokens
(≈1,500 TinyLlama tokens) per query — which still fits TinyLlama's 2,048-token window next to a short prompt and a
~250-token answer; unusually long OCR-heavy chunks are trimmed by a token budget in Part D, and the weaker rank-1 accuracy
of large chunks (0/10 vs 3/10) is what reranking in Part C should recover.

In [ ]:
CHOSEN_SIZE = 400

TINYLLAMA_CONTEXT     = 2048
PROMPT_ANSWER_BUDGET  = 150 + 250    # instructions + question, plus the generated answer
row = a2_table.set_index("Chunk size (tokens)").loc[CHOSEN_SIZE]

print(f"Chosen chunk size: {CHOSEN_SIZE} tokens")
for label, n in [("average", row["TinyLlama tokens per query (avg)"]), ("worst ", row["TinyLlama tokens per query (max)"])]:
    total = n + PROMPT_ANSWER_BUDGET
    print(f"  {label} query: {n:>6.0f} context + {PROMPT_ANSWER_BUDGET} prompt/answer = {total:>6.0f} TinyLlama tokens "
          f"({100 * total / TINYLLAMA_CONTEXT:.0f}% of {TINYLLAMA_CONTEXT}) -> {'fits' if total <= TINYLLAMA_CONTEXT else 'needs trimming in Part D'}")

CHUNKS = chunk_sets[CHOSEN_SIZE]
dense  = dense_indexes[CHOSEN_SIZE]

### A1.2 — Dense search report for the final chunk set (1 mark)

In [ ]:
a1_report = pd.DataFrame([
    ("Embedding model", EMBED_MODEL_ID),
    ("Max input length (tokens)", MAX_TOKENS),
    ("Vector size", EMBED_DIM),
    ("Similarity measure", "Cosine similarity (L2-normalised vectors + inner product, FAISS IndexFlatIP, exact search)"),
    ("Batch size", EMBED_BATCH),
    ("Device", DEVICE),
    ("Final chunk set", f"{CHOSEN_SIZE}-token fixed chunks, no overlap, {len(CHUNKS):,} chunks"),
    ("Index build time (s)", f"{dense.build_seconds:.2f}  (embedding {dense.embed_seconds:.2f} + index add {dense.add_seconds:.4f})"),
], columns=["Item", "Value"])
a1_report

,Item,Value
0,Embedding model,BAAI/bge-small-en-v1.5
1,Max input length (tokens),512
2,Vector size,384
3,Similarity measure,"Cosine similarity (L2-normalised vectors + inner product, FAISS IndexFlatIP, exact search)"
4,Batch size,64
5,Device,cuda
6,Final chunk set,"400-token fixed chunks, no overlap, 1,601 chunks"
7,Index build time (s),13.90 (embedding 13.89 + index add 0.0004)


> **Observation:** almost all of the build time is spent *embedding* the chunks (a forward pass of the model per chunk);
> adding vectors to a flat FAISS index is just a memory copy. The flat index is exact; at this corpus size (a few thousand
> vectors) approximate indexes such as HNSW or IVF are unnecessary.

---
# Part B — Keyword and hybrid search (4 marks)

All three methods search the same 1,601 × 400-token chunks, return the **top 5**, and are scored on the same 10 queries.

| Method | Matches on | Strong when | Weak when |
|---|---|---|---|
| **Dense** (BGE + FAISS) | meaning (vectors) | the query paraphrases the text ("awkward posture" ≈ "bending and reaching") | exact names, numbers, codes; diluted long chunks |
| **BM25** (keywords) | exact word overlap, weighted by rarity | rare, specific terms ("Type A", "ISO 9002", "cascading") | synonyms, paraphrases, OCR-mangled words |
| **Hybrid (RRF)** | ranks from both lists | the two methods make *different* mistakes | both methods miss the same chunk |

### Shared evaluation harness
`evaluate()` runs one warm-up query (not timed), then times each of the 10 queries end-to-end (query in → top 5 out) and
records the rank of the first chunk that contains the answer span.

In [20]:
TOP_K = 5

def first_hit_rank(results, span):
    return next((r["rank"] for r in results if is_hit(r["text"], span)), None)

def evaluate(search_fn, k=TOP_K):
    search_fn(QUERIES[0], k)                                   # warm-up run, not timed
    per_query, latencies = [], []
    for e in EVAL:
        t0 = time.perf_counter()
        results = search_fn(e["query"], k)
        latencies.append((time.perf_counter() - t0) * 1000)
        per_query.append({"qid": e["qid"], "hit_rank": first_hit_rank(results, e["span"]), "results": results})
    ranks = [p["hit_rank"] for p in per_query]
    summary = {"Average query latency (ms)": round(float(np.mean(latencies)), 2),
               "Hit rate@1 (%)": 100 * np.mean([r == 1 for r in ranks]),
               "Hit rate@3 (%)": 100 * np.mean([r is not None and r <= 3 for r in ranks]),
               "Hit rate@5 (%)": 100 * np.mean([r is not None for r in ranks])}
    return summary, per_query

b_summary, b_per_query = {}, {}

## B1. Dense (1 mark)
The Part A index, unchanged, now asked for the top 5.

In [21]:
def dense_search(query, k=TOP_K):
    return dense.search(query, k)

b_summary["Dense"], b_per_query["Dense"] = evaluate(dense_search)
b_summary["Dense"]

{'Average query latency (ms)': 12.67,
 'Hit rate@1 (%)': np.float64(0.0),
 'Hit rate@3 (%)': np.float64(60.0),
 'Hit rate@5 (%)': np.float64(70.0)}

## B2. BM25 (1 mark)
**BM25** scores a chunk by summing, over each query word, *how rare the word is* (IDF) × *how often it appears in the chunk*
(TF), with two corrections:
* **TF saturation (`k1 = 1.5`)** — the 10th "maintenance" in a chunk adds far less than the 1st.
* **Length normalisation (`b = 0.75`)** — long chunks are penalised for containing more words by chance.

Tokenisation: lowercase, split on non-alphanumerics (so `ISO 9001` → `iso`, `9001`), and drop common English stopwords,
which carry no topical signal. Note that OCR damage hurts BM25 directly: `management-ied` becomes `management` + `ied`,
and `led` will never match it.

In [22]:
from rank_bm25 import BM25Okapi

STOPWORDS = set("""a an and are as at be been being but by can could did do does for from had has have how i if in into is
it its may might must not of on or our should so such than that the their them then there these they this those to was
we were what when where which while who why will with would you your""".split())

def bm25_tokenize(text: str):
    return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOPWORDS]

class BM25Index:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        t0 = time.perf_counter()
        self.bm25 = BM25Okapi([bm25_tokenize(c["text"]) for c in chunks], k1=k1, b=b)
        self.build_seconds = time.perf_counter() - t0

    def search(self, query: str, k: int = TOP_K):
        scores = self.bm25.get_scores(bm25_tokenize(query))
        top = np.argsort(-scores)[:k]
        return [{**self.chunks[i], "score": float(scores[i]), "rank": r} for r, i in enumerate(top, start=1)]

bm25 = BM25Index(CHUNKS)
print(f"BM25 index built over {len(CHUNKS):,} chunks in {bm25.build_seconds:.2f} s")
print("Query tokens for Q8:", bm25_tokenize(EVAL[7]["query"]))

def bm25_search(query, k=TOP_K):
    return bm25.search(query, k)

b_summary["BM25"], b_per_query["BM25"] = evaluate(bm25_search)
b_summary["BM25"]

BM25 index built over 1,601 chunks in 0.42 s
Query tokens for Q8: ['common', 'elements', 'included', 'iso', '9001', 'iso', '9002', 'iso', '9003', 'quality', 'system', 'models']


{'Average query latency (ms)': 4.42,
 'Hit rate@1 (%)': np.float64(10.0),
 'Hit rate@3 (%)': np.float64(30.0),
 'Hit rate@5 (%)': np.float64(50.0)}

## B3. Hybrid with Reciprocal Rank Fusion (1 mark)
Dense scores (cosine, ~0.5–0.9) and BM25 scores (unbounded, ~5–40) are on different scales, so adding them is meaningless.
**RRF uses only ranks:**

$$\text{RRF}(d) = \frac{1}{60 + \text{rank}_\text{dense}(d)} + \frac{1}{60 + \text{rank}_\text{BM25}(d)}$$

We fuse the **top 20** of each list (a chunk missing from a list gets 0 from it) and keep the top 5. The constant 60
flattens the curve: rank 1 is worth 1/61 ≈ 0.0164 and rank 20 still 1/80 = 0.0125, so a chunk ranked fairly well by
*both* methods (e.g. 4th and 6th → 0.0156 + 0.0152 = 0.0308) beats a chunk ranked 1st by only one (0.0164).

In [23]:
RRF_K, FUSE_DEPTH = 60, 20

def rrf_fuse(ranked_lists: dict, k=TOP_K, rrf_k=RRF_K):
    # ranked_lists: {"dense": [...], "bm25": [...]}, each a list of results with 1-based "rank"
    fused = {}
    for method, results in ranked_lists.items():
        for r in results:
            entry = fused.setdefault(r["chunk_id"], {**r, "score": 0.0, "dense_rank": None, "bm25_rank": None})
            entry["score"] += 1.0 / (rrf_k + r["rank"])
            entry[f"{method}_rank"] = r["rank"]
    best = sorted(fused.values(), key=lambda x: x["score"], reverse=True)[:k]
    return [{**r, "rank": i} for i, r in enumerate(best, start=1)]

def hybrid_search(query, k=TOP_K):
    return rrf_fuse({"dense": dense.search(query, FUSE_DEPTH), "bm25": bm25.search(query, FUSE_DEPTH)}, k)

b_summary["Hybrid (RRF)"], b_per_query["Hybrid (RRF)"] = evaluate(hybrid_search)

# Worked example: where did the fused top 5 come from?
e = EVAL[4]
print(f"Q{e['qid']}: {e['query']}\n")
pd.DataFrame([{"fused rank": r["rank"], "RRF score": round(r["score"], 4), "dense rank": r["dense_rank"],
               "BM25 rank": r["bm25_rank"], "doc": r["doc"], "contains span": is_hit(r["text"], e["span"])}
              for r in hybrid_search(e["query"])])

Q5: What is the difference between reactive maintenance and more advanced maintenance strategies?



,fused rank,RRF score,dense rank,BM25 rank,doc,contains span
0,1,0.0325,2,1,NIST_Manufacturing_Machinery_Maintenance.txt,False
1,2,0.0320,1,4,NIST_Manufacturing_Machinery_Maintenance.txt,False
2,3,0.0310,3,6,NIST_Manufacturing_Machinery_Maintenance.txt,False
3,4,0.0306,8,3,NIST_Manufacturing_Machinery_Maintenance.txt,False
4,5,0.0303,5,7,NIST_Manufacturing_Machinery_Maintenance.txt,False


### Search comparison table

In [24]:
b_table = pd.DataFrame(b_summary).T.rename_axis("Method").reset_index()
b_table

,Method,Average query latency (ms),Hit rate@1 (%),Hit rate@3 (%),Hit rate@5 (%)
0,Dense,12.67,0.0,60.0,70.0
1,BM25,4.42,10.0,30.0,50.0
2,Hybrid (RRF),15.62,20.0,60.0,70.0


In [25]:
# Per query: rank (1-5) of the first chunk containing the answer span, "-" = not in top 5
pd.DataFrame({m: [p["hit_rank"] or "-" for p in rows] for m, rows in b_per_query.items()},
             index=[f"Q{e['qid']}: {e['query'][:60]}" for e in EVAL])

,Dense,BM25,Hybrid (RRF)
Q1: How did Malcolm Baldrige quality concepts influence the STEP,5,2,3
Q2: How can workstation design reduce excessive reaching and awk,3,-,4
Q3: What three kinds of evidence can be used to assess results u,2,5,1
Q4: Why are statistical methods useful when analysing manufactur,-,-,-
Q5: What is the difference between reactive maintenance and more,-,-,-
Q6: What costs are included when analysing direct and indirect m,2,-,2
Q7: What is the difference between Type A and Type B evaluations,3,5,3
"Q8: What common elements are included in the ISO 9001, ISO 9002,",2,2,2
Q9: Why is management involvement important in a quality-improve,3,1,1
Q10: What is the main purpose of machine guarding?,-,-,-


In [26]:
def show_top3(qid, methods=("Dense", "BM25", "Hybrid (RRF)"), width=220):
    # Side-by-side look at one query: the top-3 chunks of each method, marked HIT if they contain the span.
    e = EVAL[qid - 1]
    print(f"Q{qid}: {e['query']}\nSpan: {e['span']}\n")
    for m in methods:
        print(f"── {m} " + "─" * 60)
        for r in b_per_query[m][qid - 1]["results"][:3]:
            flag = "HIT " if is_hit(r["text"], e["span"]) else "    "
            print(f" {flag}#{r['rank']} [{r['doc']}] {r['text'][:width]}...")
        print()

show_top3(5)

Q5: What is the difference between reactive maintenance and more advanced maintenance strategies?
Span: is maintenance done, typically, after equipment has failed to produce a product within desired quality or production targets

── Dense ────────────────────────────────────────────────────────────
     #1 [NIST_Manufacturing_Machinery_Maintenance.txt] Advanced Maintenance Techniques The perceived benefit of potentially adopting some amount of predictive maintenance was $6.5 billion from downtime reduction and $67.3 billion in increased sales. Other benefits such as re...
     #2 [NIST_Manufacturing_Machinery_Maintenance.txt] Survey in Section 3. Direct maintenance costs are discussed in Section 4. Section 5 discusses losses, including downtime, lost sales, and defects along with additional losses and injuries due to inadequate maintenance st...
     #3 [NIST_Manufacturing_Machinery_Maintenance.txt] , 35 % fewer lost sales due to defects from maintenance, and 29 % less lost sales due t

## B4. Which method works best, and why (1 mark)
> ✍️ **TODO (write after running):** two or three sentences. Name the best method from the table (look at hit rate@1 and @3
> first, then latency), explain *why* using what each method matches on, and support it with one query from the per-query
> table — ideally one where the methods disagree. Use `show_top3(qid)` to see the actual chunks for that query.